# 5. Demo Pipeline

A complete pipeline in one notebook: **download** raw volumes from three public
networks, **archive** them, **plot** them, and cut a **vertical cross-section**.

| radar | network | format | reader |
|---|---|---|---|
| `KDVN` | NEXRAD (US) — Davenport, Iowa | Level II | `open_nexradlevel2_datatree` |
| `FANJ` | FMI (Finland) — Anjalankoski | ODIM HDF5 | `open_odim_datatree` |
| `GUA` | IDEAM (Colombia) — Guaviare | IRIS/Sigmet | `open_iris_datatree` |

All three are public and need no credentials, and the cases below are only the
defaults — section 1 is where you pick a different time.

In [ ]:
import shutil
import tarfile
import urllib.parse
import urllib.request
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import xradar

import raddb
from raddb.lut import suggest_crs

print("raddb", raddb.__version__, "| xradar", xradar.__version__)

## Configuration

`RAW_DIR` is where the downloaded files land, `ARCHIVE_DIR` is the archive the
other tutorials use — the three radars below are simply added to it.

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these two paths to point at your own machine
# --------------------------------------------------------------------------
RAW_DIR     = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/tutorial_raw").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

RAW_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    # NEXRAD is mirrored on Google Cloud as one tar per radar per hour.
    "KDVN": {"kind": "nexrad", "site": "KDVN", "open": xradar.io.open_nexradlevel2_datatree},
    # FMI publishes one ODIM PVOL per volume.  The ODIM code is "fianj" — five
    # characters, where a RadDB radar name is at most four, so it is aliased.
    "FANJ": {"kind": "odim", "site": "fianj", "open": xradar.io.open_odim_datatree},
    # IDEAM splits a volume across task files: SURVP (0.5 deg), PRECA (1.5-5.1),
    # PRECB, PRECC.  One task is one DataTree.
    "GUA": {"kind": "iris", "site": "Guaviare", "open": xradar.io.open_iris_datatree},
}

print("raw files:", RAW_DIR)
print("archive  :", ARCHIVE_DIR)

## 1. Which volume?

**This is the cell to edit to run the demo on other weather.** One timestamp per
radar, UTC — the networks publish on their own days, so they are picked
independently. Each is resolved to the volume **at or after** the time asked for.

What each archive holds, if you go looking for another case:

* **NEXRAD** — the Google mirror is complete for 2024; one tar per radar per
  hour, ~10 volumes inside, so any hour of any day works.
* **FMI** — one file every **5 minutes**, so the timestamp is rounded down to
  the 5-minute mark.
* **IDEAM** — tasks cycle `SURVP` → `PRECA` → `PRECB` → `PRECC` every few
  minutes, and one task is one DataTree. Which elevations you get therefore
  depends on the minute you ask for: the default lands on `PRECA` (1.5-5.1°).

In [ ]:
# --------------------------------------------------------------------------
# WHICH TIMESTEP? — one UTC timestamp per radar, edit freely
# --------------------------------------------------------------------------
TIMES = {
    "KDVN": "2024-06-25 23:04",   
    "FANJ": "2024-08-09 12:00",   
    "GUA":  "2024-06-12 19:02",   
}

for name, when in TIMES.items():
    print(f"{name:<5} {pd.Timestamp(when):%Y-%m-%d %H:%M} UTC")

## 2. Download

In [ ]:
def resolve_url(name, source, when):
    """URL of the volume at or after *when* for one source."""
    t = pd.Timestamp(when)
    site = source["site"]

    if source["kind"] == "nexrad":
        obj = (f"{t:%Y/%m/%d}/{site}/NWS_NEXRAD_NXL2DPBL_{site}"
               f"_{t:%Y%m%d%H}0000_{t:%Y%m%d%H}5959.tar")
        return ("https://storage.googleapis.com/download/storage/v1/b/gcp-public-data-nexrad-l2/o/"
                + urllib.parse.quote(obj, safe="") + "?alt=media")

    if source["kind"] == "odim":
        t = t.floor("5min")                       # FMI publishes every 5 minutes
        return ("https://fmi-opendata-radar-volume-hdf5.s3.eu-west-1.amazonaws.com/"
                f"{t:%Y/%m/%d}/{site}/{t:%Y%m%d%H%M}_{site}_PVOL.h5")

    # IDEAM: list the first key at or after the requested second.
    prefix = f"l2_data/{t:%Y/%m/%d}/{site}/"
    query = urllib.parse.urlencode({
        "list-type": "2", "max-keys": "1", "prefix": prefix,
        "start-after": f"{prefix}{name}{t - pd.Timedelta(seconds=1):%y%m%d%H%M%S}",
    })
    with urllib.request.urlopen(f"https://s3-radaresideam.s3.amazonaws.com/?{query}", timeout=120) as response:
        listing = response.read().decode()
    if "<Key>" not in listing:
        raise RuntimeError(f"IDEAM has nothing at or after {t} for {site}")
    return "https://s3-radaresideam.s3.amazonaws.com/" + listing.split("<Key>")[1].split("</Key>")[0]


def download(url, dest_dir, when):
    """Fetch one volume into *dest_dir* and return its path."""
    if ".tar" in url:
        wanted = pd.Timestamp(when)
        with urllib.request.urlopen(url, timeout=600) as response, tarfile.open(fileobj=response, mode="r|") as tar:
            for member in tar:
                # KDVN20240625_230458_V06.ar2v — *_MDM.ar2v are metadata stubs.
                if not member.name.endswith(".ar2v") or "_MDM" in member.name:
                    continue
                stamp = pd.to_datetime(Path(member.name).stem[4:19], format="%Y%m%d_%H%M%S")
                if stamp < wanted:
                    continue
                out = dest_dir / Path(member.name).name
                with tar.extractfile(member) as src, open(out, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                return out
        raise RuntimeError(f"no volume at or after {wanted} in {url}")

    out = dest_dir / Path(urllib.parse.urlparse(url).path).name
    if not out.exists():
        urllib.request.urlretrieve(url, out)
    return out


for name, source in SOURCES.items():
    url = resolve_url(name, source, TIMES[name])
    source["path"] = download(url, RAW_DIR, TIMES[name])
    print(f"{name:<5} {source['path'].name:<32} {source['path'].stat().st_size / 1e6:6.1f} MB")

## 3. Archive

Each file is opened with its own xradar reader and handed to `archive()` — no
preparation in between. A real volume often drops a ray or two (718 of 720 on
WSR-88D, 358 of 360 here); RadDB reads that as a rotation with holes, keeps the
missing rays' slots in the LUT so a later volume that records them still joins,
and archives what is there.

The CRS is **mandatory to write** and comes from `suggest_crs()`, which returns
the UTM zone of the site: three radars on three continents, three projections.

In [ ]:
for name, source in SOURCES.items():
    dt = source["open"](str(source["path"]))
    site = dt["/"].ds
    crs = suggest_crs(latitude=float(site["latitude"]), longitude=float(site["longitude"]))
    print(f"{name}: {sum(1 for g in dt.groups if g.startswith('/sweep_'))} sweeps, "
          f"site {float(site['latitude']):.2f}, {float(site['longitude']):.2f} -> EPSG:{crs}")

    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=crs).archive(datatree=dt, radar=name)

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
db.inventory()

## 4. Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

volumes = {}
for ax, name in zip(axes, SOURCES):
    volumes[name] = db.open(radars=name)
    volumes[name].plot_ppi(sweep=1, variable="DBZH", ax=ax, coords="xy", context=True)
    ax.set_title(f"{name}")

plt.tight_layout()
plt.show()

## 5. A cross-section, drawn by hand

`interactive_crop()` puts an [ipyleaflet](https://ipyleaflet.readthedocs.io/) map
in front of you. It dispatches on the shape you draw — rectangle, polygon and
marker run the three crops, and the **polyline** runs
`extract_cross_section`, which is the one used here.

Draw a line, then click **Apply crop** before running the next cell.

In [ ]:
kdvn = volumes["KDVN"]
selector = kdvn.interactive_crop()

The drawn section is on `selector.result` — an ordinary RadDB, carrying the extra
`d_center` / `z_center` / `cs_polygon` columns that describe the curtain. Nothing
drawn (or no live kernel) falls back to a fixed line through the same storm, so
the plot below always has a section to draw.

In [ ]:
LINE = ((-91.30, 40.95), (-91.30, 42.10))    # fallback: north-south through the strongest cell

drawn = getattr(selector, "result", None)
if drawn is not None and selector.kind == "cross_section":
    section = drawn
    print(f"drawn section: {len(section):,} gates")
else:
    section = kdvn.extract_cross_section(p1=LINE[0], p2=LINE[1], crs=4326)
    print(f"nothing drawn — using the fixed line {LINE}: {len(section):,} gates")

fig, ax = plt.subplots(figsize=(9, 4.5))
section.plot_vcs(variable="DBZH", ax=ax)
ax.set_title(f"KDVN | {pd.Timestamp(TIMES['KDVN']):%Y-%m-%d %H:%M} UTC | vertical cross-section")
plt.tight_layout()
plt.show()

---

That is the whole pipeline: **download → archive → open → plot → cut**. The
archive now holds three more radars, and everything the other notebooks do —
filters, label selection, the other three crops, RHI, CAPPI — applies to them
unchanged.

**See also:** [1 — Archiving](01_archiving.ipynb) ·
[2 — Opening and filtering](02_opening_and_filtering.ipynb) ·
[3 — Areas of interest](03_area_of_interest.ipynb) ·
[4 — Plots](04_plots.ipynb)